In [1]:
# Cell 1: Install dependencies (nếu cần)
!pip install -q transformers torch faiss-cpu Pillow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [2]:
# Cell 2: Imports và Config
import os
import numpy as np
from pathlib import Path
from typing import List, Tuple
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor
import faiss

# Debug: List available input paths
import os
print("Available inputs:")
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level < 4:  # Only show first 4 levels
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 3:
            for f in files[:5]:  # Show first 5 files
                print(f"{indent}  {f}")

# Try multiple possible paths
possible_paths = [
    "/kaggle/input/chetankv/dogs-cats-images/dataset/test_set/cats",
    "/kaggle/input/dogs-cats-images/dataset/test_set/cats",
    "/kaggle/input/chetankv/dataset/test_set/cats"
]

DATASET_PATH = None
for path in possible_paths:
    if os.path.exists(path):
        DATASET_PATH = path
        print(f"\nFound dataset at: {path}")
        break

if not DATASET_PATH:
    print("\nERROR: Could not find dataset in any expected location")
    import sys
    sys.exit(1)

MODEL_ID = "google/siglip2-base-patch16-naflex"
EMBEDDING_DIM = 768
BATCH_SIZE = 16
OUTPUT_DIR = "/kaggle/working"

# Force CPU to avoid GPU compatibility issues
device = "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Cell 3: Find images
def find_images(root_path: str) -> List[Tuple[str, str]]:
    image_files = []
    extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    root = Path(root_path)

    for img_path in root.rglob("*"):
        if img_path.suffix.lower() in extensions and img_path.is_file():
            img_id = str(img_path.relative_to(root))
            image_files.append((img_id, str(img_path)))

    return image_files

print(f"Scanning images in: {DATASET_PATH}")
image_list = find_images(DATASET_PATH)
print(f"Found {len(image_list)} images")

image_ids = [img_id for img_id, _ in image_list]
image_paths = [img_path for _, img_path in image_list]

Scanning images in: /kaggle/input/datasets/chetankv/dogs-cats-images/dataset/test_set/cats/
Found 1000 images


In [4]:
# Cell 4: Load model
print(f"Loading model: {MODEL_ID}")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)
model = model.to(device)
model.eval()
print("Model loaded")

Loading model: google/siglip2-base-patch16-naflex


preprocessor_config.json:   0%|          | 0.00/393 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/329 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Model loaded


In [5]:
# Cell 5: Embed images
print(f"Embedding {len(image_paths)} images...")
all_embeddings = []

for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Embedding"):
    batch_paths = image_paths[i:i+BATCH_SIZE]
    batch_images = []

    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            batch_images.append(img)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            batch_images.append(Image.new("RGB", (224, 224), (0, 0, 0)))
        
    inputs = processor(images=batch_images, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
        image_features = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs
        image_features = F.normalize(image_features, p=2, dim=-1)

    all_embeddings.append(image_features.cpu().numpy())

if len(all_embeddings) == 0:
    print("ERROR: No images found. Check DATASET_PATH.")
    import sys
    sys.exit(1)

image_embeddings = np.vstack(all_embeddings)
print(f"Embeddings shape: {image_embeddings.shape}")

norms = np.linalg.norm(image_embeddings, axis=1)
print(f"L2 norms - Min: {norms.min():.4f}, Max: {norms.max():.4f}, Mean: {norms.mean():.4f}")

Embedding 1000 images...


Embedding: 100%|██████████| 63/63 [00:14<00:00,  4.35it/s]

Embeddings shape: (1000, 768)
L2 norms - Min: 0.9995, Max: 1.0000, Mean: 1.0000


In [6]:
np.savez(
    f"{OUTPUT_DIR}/embeddings.npz",
    embeddings=image_embeddings,
    image_ids=image_ids,
    image_paths=image_paths
)
print(f"Embeddings saved to {OUTPUT_DIR}/embeddings.npz")

index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(image_embeddings.astype(np.float32))
faiss.write_index(index, f"{OUTPUT_DIR}/faiss_index.bin")
print(f"FAISS index saved to {OUTPUT_DIR}/faiss_index.bin")

print("\n=== Complete ===")
print(f"Total images: {len(image_paths)}")
print(f"Embeddings: {os.path.getsize(f'{OUTPUT_DIR}/embeddings.npz') / 1024**2:.2f} MB")
print(f"FAISS index: {os.path.getsize(f'{OUTPUT_DIR}/faiss_index.bin') / 1024**2:.2f} MB")
print("\nDownload embeddings.npz and faiss_index.bin from Output tab")

Embeddings saved to /kaggle/working/embeddings.npz
FAISS index saved to /kaggle/working/faiss_index.bin

=== Complete ===
Total images: 1000
Embeddings: 1.83 MB
FAISS index: 2.93 MB

Download embeddings.npz and faiss_index.bin from Output tab
